In [11]:
import os
from collections import namedtuple

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from robobrowser import RoboBrowser

# Load environment variables from .env file
load_dotenv()

# Get credentials from environment variables
KICKTIPP_USERNAME = os.getenv("KICKTIPP_USERNAME")
KICKTIPP_PASSWORD = os.getenv("KICKTIPP_PASSWORD")

if not KICKTIPP_USERNAME or not KICKTIPP_PASSWORD:
    raise ValueError(
        "Please set KICKTIPP_USERNAME and KICKTIPP_PASSWORD in your .env file"
    )

plt.rcParams["font.family"] = "Noto sans"

In [12]:
def login(browser: RoboBrowser, username, password):
    browser.open("https://www.kicktipp.de/info/profil/login")
    form = browser.get_form()
    form["kennung"] = username
    form["passwort"] = password
    browser.submit_form(form)
    return browser.session.cookies["login"]


browser = RoboBrowser(parser="html5lib")
browser.session.cookies["login"] = login(browser, KICKTIPP_USERNAME, KICKTIPP_PASSWORD)

In [13]:
MatchTip = namedtuple(
    "MatchTip", "match_day match_id home_team away_team home_tip away_tip"
)


def build_tippabgabe_url(community, matchday):
    return f"https://www.kicktipp.de/{community}/tippabgabe?&spieltagIndex={matchday}"


def get_matches(browser, community, matchday):
    url = build_tippabgabe_url(community, matchday)
    browser.open(url)
    home_teams = [td.text for td in browser.select("tr > td:nth-child(2)")]
    away_teams = [td.text for td in browser.select("tr > td:nth-child(3)")]
    value_default_0 = lambda inp: int(inp.attrs["value"]) if "value" in inp.attrs else 0
    home_tips = [
        value_default_0(inp) for inp in browser.select('input[id$="_heimTipp"]')
    ]
    away_tips = [
        value_default_0(inp) for inp in browser.select('input[id$="_gastTipp"]')
    ]
    matches = []
    for i in range(len(home_teams)):
        try:
            matches.append(
                MatchTip(
                    matchday,
                    i,
                    home_teams[i],
                    away_teams[i],
                    home_tips[i],
                    away_tips[i],
                )
            )
        except:
            print(f"Match is over {matchday=} {home_teams[i]=} {away_teams[i]=}")
    return matches


def print_match(match):
    print(f"{match.home_tip}:{match.away_tip}   {match.home_team} : {match.away_team}")


def make_tipps(browser, community, match_tips):
    tips_by_match_day = {}
    for t in match_tips:
        match_day = str(t.match_day)
        if match_day not in tips_by_match_day:
            tips_by_match_day[match_day] = []
        tips_by_match_day[match_day].append(t)

    for match_day, match_day_tips in tips_by_match_day.items():
        print(f"Placing bets for matchday {match_day}")
        browser.open(build_tippabgabe_url(community, match_day))
        form = browser.get_form()
        field_home_tips = browser.select('input[id$="_heimTipp"]')
        field_away_tips = browser.select('input[id$="_gastTipp"]')
        for match in match_day_tips:
            home_field = field_home_tips[match.match_id]
            away_field = field_away_tips[match.match_id]
            form[home_field.attrs["name"]] = str(match.home_tip)
            form[away_field.attrs["name"]] = str(match.away_tip)
            print(
                f"{match.home_tip}:{match.away_tip}   {match.home_team} : {match.away_team}"
            )

        browser.submit_form(form, submit="submitbutton")

In [14]:
df_predictions = pd.read_pickle("data/predictions_poisson_2025.pkl")
df_predictions.head()

,match_day,season,league,datetime,home,away,home_goals,away_goals,home_goals_pred,away_goals_pred
id,,,,,,,,,,
77256,1,2025,bl1,2025-08-22 20:30:00,Bayern,Leipzig,6,0,2,1
77257,1,2025,bl1,2025-08-23 15:30:00,Leverkusen,Hoffenheim,1,2,2,1
77258,1,2025,bl1,2025-08-23 15:30:00,Frankfurt,Bremen,4,1,2,1
77259,1,2025,bl1,2025-08-23 15:30:00,Freiburg,Augsburg,1,3,1,0
77262,1,2025,bl1,2025-08-23 15:30:00,Union Berlin,Stuttgart,2,1,1,1


In [19]:
team_names_df = df_predictions["home"].unique().tolist()
team_names_df

['Bayern',
 'Leverkusen',
 'Frankfurt',
 'Freiburg',
 'Union Berlin',
 'Heidenheim',
 'St. Pauli',
 'Mainz',
 'Gladbach',
 'HSV',
 'Leipzig',
 'Bremen',
 'Stuttgart',
 'Hoffenheim',
 'Augsburg',
 'Wolfsburg',
 'Dortmund',
 'Köln']

In [23]:
from difflib import SequenceMatcher


def similar(name_1, name_2):
    return SequenceMatcher(None, name_1, name_2).ratio()


def find_df_name(kicktipp_name):
    if "Hamburg" in kicktipp_name:
        kicktipp_name = "HSV"
    similarities = [similar(kicktipp_name, df_name) for df_name in team_names_df]
    max_similarity = max(similarities)
    if max_similarity < 0.5:
        raise ValueError(f"No similar team name found for {kicktipp_name}")
    max_index = similarities.index(max_similarity)
    return team_names_df[max_index]

In [ ]:
def find_prediction(match):
    home = find_df_name(match.home_team)
    away = find_df_name(match.away_team)
    row = df_predictions[
        (df_predictions["home"] == home) & (df_predictions["away"] == away)
    ]
    assert row.shape[0] == 1, (
        f"Found {row.shape[0]} predictions for {match.home_team} vs {match.away_team}"
    )
    home_tip = int(row["home_goals_pred"].iloc[0])
    away_tip = int(row["away_goals_pred"].iloc[0])

    return MatchTip(
        match.match_day,
        match.match_id,
        match.home_team,
        match.away_team,
        home_tip,
        away_tip,
    )


communities = ["mark-forsters-tipp-tafelrunde"]
community_predictions = {}
for community in communities:
    all_predictions = []
    for match_day in range(1, 35):
        matches = get_matches(browser, community, match_day)
        predictions = [find_prediction(m) for m in matches]
        all_predictions += predictions
        make_tipps(browser, community, predictions)
    community_predictions[community] = all_predictions

Match is over matchday=1 home_teams[i]='1. FC Union Berlin' away_teams[i]='VfB Stuttgart'
Match is over matchday=1 home_teams[i]='1. FC Heidenheim 1846' away_teams[i]='VfL Wolfsburg'
Match is over matchday=1 home_teams[i]='Bayer 04 Leverkusen' away_teams[i]='1899 Hoffenheim'
Match is over matchday=1 home_teams[i]='SC Freiburg' away_teams[i]='FC Augsburg'
Match is over matchday=1 home_teams[i]='Eintracht Frankfurt' away_teams[i]='Werder Bremen'
Match is over matchday=1 home_teams[i]='FC St. Pauli' away_teams[i]='Borussia Dortmund'
Match is over matchday=1 home_teams[i]='FSV Mainz 05' away_teams[i]='1. FC Köln'
Match is over matchday=1 home_teams[i]='Bor. Mönchengladbach' away_teams[i]='Hamburger SV'
Mapping FC Bayern München to Bayern, RB Leipzig to Leipzig
Placing bets for matchday 1
2:1   FC Bayern München : RB Leipzig
Mapping Hamburger SV to HSV, FC St. Pauli to St. Pauli
Mapping RB Leipzig to Leipzig, 1. FC Heidenheim 1846 to Heidenheim
Mapping VfB Stuttgart to Stuttgart, Bor. Mönch